# Notebook 6: Fine-tuning Pretrained GloVe Embeddings (SOLUTION)

*Module 2 (Text Similarity). ML & NLP by Data Trainers LLC.*

This is the **solution** notebook. Every `YOUR CODE` placeholder is filled in, with extra commentary on the *why* and on common mistakes.

## The Scenario

In **Notebook 5** you trained a CBOW model from scratch on 5,000 Yelp reviews. You got vectors that knew **pizza** is similar to **burger**, but they had no sense of **king**, **queen**, **paris**, or **france**, because those words barely appear in restaurant reviews.

## The Plan

Use Stanford's **GloVe 6B 100d** vectors (trained on 6 billion tokens of Wikipedia + Gigaword) and apply three classic recipes:

1. **Variant A, random init**: the NB5 baseline.
2. **Variant B, frozen GloVe**: load GloVe, lock it (`freeze=True`).
3. **Variant C, fine-tuned GloVe**: load GloVe, allow updates (`freeze=False`).

**Runtime**: ~45 minutes on Colab free tier.

**Framework**: PyTorch.

## Section 0: Environment Setup

In [ ]:
# Install required packages (skip if you already have them locally)
!pip install -q torch scikit-learn pandas numpy matplotlib gensim textblob

In [ ]:
# Core imports
import os, re, random, math, time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import gensim
from textblob import TextBlob

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")
print("Environment ready.")

In [ ]:
EMBEDDING_DIM = 100        # MUST match the GloVe file (100d)
CORPUS_SIZE   = 5000
WINDOW_SIZE   = 2
BATCH_SIZE    = 128
EPOCHS        = 5
LR_FROZEN     = 1e-3
LR_FINETUNE   = 5e-4
MIN_FREQ      = 2
PAD_IDX       = 0

print(f"EMBEDDING_DIM={EMBEDDING_DIM}  CORPUS_SIZE={CORPUS_SIZE}  WINDOW_SIZE={WINDOW_SIZE}")
print(f"BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}  LR_FROZEN={LR_FROZEN}  LR_FINETUNE={LR_FINETUNE}")

## Section 1: Transfer Learning for NLP

ImageNet pretraining for vision; GloVe / Word2Vec / BERT for language. Same idea: someone else trained on a massive corpus, you get the result for free.

| Year | Model | Idea |
|------|-------|------|
| 2013 | Word2Vec | Predict context from center / center from context. |
| 2014 | **GloVe** | Factorize a global word-word co-occurrence matrix. |
| 2018+ | BERT, GPT | Contextual embeddings (Module 3). |

Why pretraining works: language has universal patterns (*king is to man as queen is to woman*, *paris is to france as rome is to italy*). Wikipedia + Gigaword saw billions of examples. Your Yelp corpus saw none.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
axes[0].imshow(np.random.randn(20, EMBEDDING_DIM), cmap='RdBu', aspect='auto')
axes[0].set_title('NB5: random init (white noise)')
axes[0].set_xlabel('embedding dim'); axes[0].set_ylabel('word id')
axes[1].imshow(np.random.randn(20, EMBEDDING_DIM) * 0.4, cmap='RdBu', aspect='auto')
axes[1].set_title('NB6: GloVe init (structured)')
axes[1].set_xlabel('embedding dim')
plt.tight_layout(); plt.show()

## Section 2: Loading GloVe

GloVe 6B 100d ships as a plain text file: `word v1 v2 ... v100` per line, 400K lines.

In [ ]:
GLOVE_PATH = './glove.6B.100d.txt'
if not os.path.exists(GLOVE_PATH):
    !wget -q -O ./glove.6B.100d.txt 'https://www.dropbox.com/s/dl1vswq2sz5f1ws/glove.6B.100d.txt?dl=1'
print(f"GloVe file size: {os.path.getsize(GLOVE_PATH) / 1e6:.1f} MB")

with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for _ in range(2):
        line = f.readline()
        word, rest = line.split(maxsplit=1)
        print(f"word={word!r}   first 5 dims=[{', '.join(rest.split()[:5])} ...]")

### Demo: parse the GloVe file into a Python dict

In [ ]:
def load_glove(path):
    """Parse a GloVe text file into {word: np.float32 array of shape (D,)}."""
    glove = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            word, rest = line.split(maxsplit=1)
            glove[word] = np.fromstring(rest, dtype='float32', sep=' ')
    return glove

t0 = time.time()
glove = load_glove(GLOVE_PATH)
print(f"Loaded {len(glove):,} word vectors in {time.time() - t0:.1f}s")
print(f"Each vector has shape: {glove['king'].shape} (dtype={glove['king'].dtype})")

In [ ]:
print("glove['king'][:8]  =", np.round(glove['king'][:8], 3))
print("glove['queen'][:8] =", np.round(glove['queen'][:8], 3))
print(f"\n||king||  = {np.linalg.norm(glove['king']):.3f}")
print(f"||queen|| = {np.linalg.norm(glove['queen']):.3f}")
k, q = glove['king'], glove['queen']
cos_kq = np.dot(k, q) / (np.linalg.norm(k) * np.linalg.norm(q))
print(f"\ncosine(king, queen) = {cos_kq:.3f}")

### Demo: the famous analogy *king − man + woman ≈ queen*

In [ ]:
GLOVE_WORDS = list(glove.keys())
GLOVE_MATRIX = np.stack([glove[w] for w in GLOVE_WORDS])
GLOVE_NORMS = np.linalg.norm(GLOVE_MATRIX, axis=1, keepdims=True)

def nearest_neighbors(vec, topn=5, exclude=()):
    vec = vec.astype('float32')
    vec_norm = np.linalg.norm(vec) + 1e-9
    sims = (GLOVE_MATRIX @ vec) / (GLOVE_NORMS.squeeze() * vec_norm)
    top_idx = np.argpartition(-sims, topn + len(exclude))[:topn + len(exclude)]
    top_idx = top_idx[np.argsort(-sims[top_idx])]
    out = []
    for i in top_idx:
        w = GLOVE_WORDS[i]
        if w in exclude:
            continue
        out.append((w, float(sims[i])))
        if len(out) == topn:
            break
    return out

analogy_vec = glove['king'] - glove['man'] + glove['woman']
print("king - man + woman -> nearest:")
for w, s in nearest_neighbors(analogy_vec, topn=5, exclude={'king', 'man', 'woman'}):
    print(f"   {w:<15s} sim={s:.3f}")

### Lab 2.1 (SOLUTION): Country-Capital analogy

In [ ]:
# Solution: Lab 2.1 -- country-capital analogy
# 1. Compute the analogy vector: paris - france + italy
analogy = glove['paris'] - glove['france'] + glove['italy']

# 2. Get the top-5 nearest neighbors, excluding the input words
neighbors = nearest_neighbors(analogy, topn=5, exclude={'paris', 'france', 'italy'})

# Verification
print("paris - france + italy -> nearest:")
for w, s in neighbors:
    print(f"   {w:<15s} sim={s:.3f}")

# Explanation:
# Vector arithmetic on word embeddings captures relations: the difference (paris - france)
# encodes "capital_of". Add it to italy and you should land near rome.
# Common mistake: forgetting to exclude the input words. Without exclude=..., you'd often
# see 'paris' or 'italy' itself top the list because the analogy is a small perturbation.

## Section 3: Building the Aligned Embedding Matrix

We need to bridge the GloVe `dict` and a single PyTorch tensor of shape `(vocab_size, embedding_dim)`. Row 0 is `<pad>` (zeros), other rows get GloVe vectors when available, OOV gets small random init.

In [ ]:
YELP_PATH = './yelp.csv'
if not os.path.exists(YELP_PATH):
    !wget -q -O ./yelp.csv 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(YELP_PATH)
yelp_extreme = yelp[(yelp.stars == 5) | (yelp.stars == 1)]
print(f"Total Yelp 1/5 star reviews: {len(yelp_extreme):,}")

def tokenize(text):
    return [w.lower() for w in TextBlob(text).words]

raw_texts = yelp_extreme['text'].astype(str).tolist()
corpus_tokens = []
for txt in raw_texts:
    toks = tokenize(txt)
    if len(toks) > 3:
        corpus_tokens.append(toks)
    if len(corpus_tokens) >= CORPUS_SIZE:
        break
print(f"Corpus size: {len(corpus_tokens):,} reviews")
print(f"First review (first 12 tokens): {corpus_tokens[0][:12]}")

In [ ]:
counter = Counter()
for toks in corpus_tokens:
    counter.update(toks)

PAD_TOKEN, UNK_TOKEN = '<pad>', '<unk>'
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for tok, freq in counter.most_common():
    if freq >= MIN_FREQ:
        vocab[tok] = len(vocab)
vocab_size = len(vocab)
id_to_word = {i: w for w, i in vocab.items()}
print(f"Vocabulary size (incl. <pad> and <unk>): {vocab_size:,}")
print(f"Sample first 10 entries: {list(vocab.items())[:10]}")

### Demo: build the aligned embedding matrix and report coverage

In [ ]:
rng = np.random.RandomState(SEED)
demo_matrix = rng.normal(0, 0.1, size=(vocab_size, EMBEDDING_DIM)).astype('float32')
demo_matrix[PAD_IDX] = 0.0

hits, misses = 0, 0
for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    vec = glove.get(word)
    if vec is not None:
        demo_matrix[idx] = vec
        hits += 1
    else:
        misses += 1
coverage = hits / (hits + misses)
print(f"GloVe coverage: {hits:,}/{hits + misses:,} = {coverage:.1%}")
print(f"OOV (kept as small random): {misses:,} words")
print(f"Embedding matrix shape: {demo_matrix.shape}, dtype: {demo_matrix.dtype}")
print(f"Row 0 (PAD) is all zero? {np.all(demo_matrix[0] == 0)}")

### Lab 3.1 (SOLUTION): Implement `build_embedding_matrix`

In [ ]:
# Solution: Lab 3.1 -- clean reusable function
def build_embedding_matrix(vocab, glove, embedding_dim, pad_idx=0, seed=42):
    """Returns (matrix, hits, misses).
    Row pad_idx is all zeros; words present in GloVe get their GloVe vector;
    OOV words get small random init from N(0, 0.1).
    """
    # 1. Allocate a (vocab_size, embedding_dim) float32 matrix from N(0, 0.1)
    rng = np.random.RandomState(seed)
    V = len(vocab)
    matrix = rng.normal(0, 0.1, size=(V, embedding_dim)).astype('float32')

    # 2. Set the padding row to zeros (required by nn.Embedding(padding_idx=...))
    matrix[pad_idx] = 0.0

    # 3. Loop over vocab; overwrite rows for words present in glove
    hits, misses = 0, 0
    for word, idx in vocab.items():
        if idx == pad_idx:
            continue  # never overwrite the pad row
        vec = glove.get(word)
        if vec is not None:
            matrix[idx] = vec
            hits += 1
        else:
            misses += 1
    return matrix, hits, misses

# Build it and verify
embedding_matrix, hits, misses = build_embedding_matrix(vocab, glove, EMBEDDING_DIM, pad_idx=PAD_IDX, seed=SEED)
print(f"Shape: {embedding_matrix.shape}, dtype: {embedding_matrix.dtype}")
print(f"Coverage: {hits} / {hits + misses} = {hits / max(1, hits + misses):.1%}")
print(f"Row 0 zero? {np.all(embedding_matrix[0] == 0)}")

embedding_matrix_t = torch.FloatTensor(embedding_matrix)
print(f"Torch tensor shape: {embedding_matrix_t.shape}, dtype: {embedding_matrix_t.dtype}")

# Explanation:
# Common mistakes:
# - Initializing OOV with zeros instead of small random. Identical zero rows mean every OOV
#   word looks the same to the model -- it has no way to differentiate them.
# - Forgetting matrix[pad_idx] = 0. nn.Embedding(padding_idx=0) does NOT auto-zero the row;
#   it only zeros the gradient. If row 0 has random values, padding tokens silently leak
#   noise into the masked mean.
# - Using float64 instead of float32. PyTorch will silently cast (or error on some ops);
#   either way you waste 2x memory.

## Section 4: Three CBOW Variants

In [ ]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pretrained_matrix=None, freeze=True, padding_idx=PAD_IDX):
        super().__init__()
        if pretrained_matrix is None:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
            with torch.no_grad():
                self.embedding.weight[padding_idx].zero_()
        else:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_matrix, freeze=freeze, padding_idx=padding_idx
            )
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_ids):
        e = self.embedding(context_ids)
        ctx = e.mean(dim=1)
        return self.linear(ctx)

tmp = CBOW(vocab_size, EMBEDDING_DIM).to(device)
n = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
print(f"CBOW (random init) trainable params: {n:,}")
del tmp

### CBOW data: (context, target) pairs

In [ ]:
def build_cbow_pairs(corpus_tokens, vocab, window=WINDOW_SIZE, pad_idx=PAD_IDX, unk_idx=1):
    pairs = []
    ctx_len = 2 * window
    for toks in corpus_tokens:
        ids = [vocab.get(t, unk_idx) for t in toks]
        L = len(ids)
        for i, target in enumerate(ids):
            ctx = []
            for j in range(i - window, i + window + 1):
                if j == i or j < 0 or j >= L:
                    continue
                ctx.append(ids[j])
            while len(ctx) < ctx_len:
                ctx.append(pad_idx)
            pairs.append((ctx, target))
    return pairs

pairs = build_cbow_pairs(corpus_tokens, vocab)
print(f"Total CBOW pairs: {len(pairs):,}")
ctx, tgt = pairs[5]
print(f"Sample pair: ctx={ctx} target={tgt}")
print(f"  decoded context: {[id_to_word[i] for i in ctx]}")
print(f"  decoded target : {id_to_word[tgt]}")

In [ ]:
class CBOWDataset(Dataset):
    def __init__(self, pairs):
        self.X = torch.tensor([p[0] for p in pairs], dtype=torch.long)
        self.y = torch.tensor([p[1] for p in pairs], dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_ds = CBOWDataset(pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Train loader: {len(train_loader)} batches of size {BATCH_SIZE}")

### Demo: training loop helper

In [ ]:
def train_one_variant(model, loader, lr, epochs=EPOCHS, label='model'):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for ep in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            total += loss.item()
            n += 1
        ep_loss = total / max(1, n)
        losses.append(ep_loss)
        print(f"[{label}] epoch {ep}/{epochs} | loss={ep_loss:.4f}")
    return losses

### Lab 4.1 (SOLUTION): Build and train the three variants

In [ ]:
# Solution: Lab 4.1 -- build the three variants
# We seed before each construction so the Linear layer's random init is identical
# across variants. Only the embedding init differs.
torch.manual_seed(SEED)
model_random = CBOW(vocab_size, EMBEDDING_DIM).to(device)

torch.manual_seed(SEED)
model_frozen = CBOW(vocab_size, EMBEDDING_DIM,
                    pretrained_matrix=embedding_matrix_t, freeze=True).to(device)

torch.manual_seed(SEED)
model_ft = CBOW(vocab_size, EMBEDDING_DIM,
                pretrained_matrix=embedding_matrix_t, freeze=False).to(device)

for name, m in [('random', model_random), ('frozen', model_frozen), ('finetune', model_ft)]:
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in m.parameters())
    print(f"{name:>8s} | trainable={n_train:>10,d} | total={n_total:>10,d}")

# Explanation:
# - 'frozen' has the smallest trainable count: only the Linear head trains, the
#   (vocab_size * 100) embedding parameters are locked.
# - 'random' and 'finetune' have identical trainable counts -- both train everything.
# - The `total` is the same for all three; only `trainable` differs.

In [ ]:
# Solution: Lab 4.1 (continued) -- train all three variants
losses_random   = train_one_variant(model_random, train_loader, LR_FROZEN,   label='random')
losses_frozen   = train_one_variant(model_frozen, train_loader, LR_FROZEN,   label='frozen')
losses_finetune = train_one_variant(model_ft,     train_loader, LR_FINETUNE, label='finetune')

# Explanation:
# Why the different LRs:
# - LR_FROZEN (1e-3): used for both the random and frozen variants. In random init the
#   embedding starts from noise so it needs aggressive updates; in frozen the embedding
#   doesn't update at all (only the Linear head does), so a higher LR is fine.
# - LR_FINETUNE (5e-4): used for the fine-tuned variant. The pretrained GloVe vectors
#   are valuable; large updates would 'forget' GloVe's general-purpose knowledge.
#   In production you'd often go even smaller (1e-5 to 1e-4) for fine-tuning.

### Demo: compare loss curves

In [ ]:
plt.figure(figsize=(7.5, 4))
epochs_axis = list(range(1, EPOCHS + 1))
plt.plot(epochs_axis, losses_random,   'o-', label='A: random init',     color='#FF6B6B')
plt.plot(epochs_axis, losses_frozen,   's-', label='B: frozen GloVe',    color='#4ECDC4')
plt.plot(epochs_axis, losses_finetune, '^-', label='C: fine-tuned GloVe', color='#5B8DEF')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss')
plt.title('CBOW: same architecture, three embedding inits')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

### Demo: `most_similar` per variant

In [ ]:
def model_to_keyedvectors(model, vocab, id_to_word, path):
    weights = model.embedding.weight.detach().cpu().numpy()
    V, D = weights.shape
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f"{V - 1} {D}\n")
        for i in range(1, V):
            w = id_to_word[i]
            vec = ' '.join(f"{x:.6f}" for x in weights[i])
            f.write(f"{w} {vec}\n")
    return gensim.models.KeyedVectors.load_word2vec_format(path, binary=False)

kv_random   = model_to_keyedvectors(model_random, vocab, id_to_word, './kv_random.txt')
kv_frozen   = model_to_keyedvectors(model_frozen, vocab, id_to_word, './kv_frozen.txt')
kv_finetune = model_to_keyedvectors(model_ft,     vocab, id_to_word, './kv_finetune.txt')
print("KeyedVectors built for all three variants.")

In [ ]:
def show_neighbors(label, kv, word, topn=5):
    if kv is None or word not in kv:
        print(f"[{label}] '{word}' not in vocab -- skipping")
        return
    print(f"[{label}] most_similar('{word}')")
    for w, s in kv.most_similar(positive=[word], topn=topn):
        print(f"   {w:<20s} sim={s:.3f}")
    print()

for word in ['cheap', 'pizza', 'service']:
    print("=" * 60)
    show_neighbors('A: random  ', kv_random,   word)
    show_neighbors('B: frozen  ', kv_frozen,   word)
    show_neighbors('C: finetune', kv_finetune, word)

### When to freeze, when to fine-tune?

| Situation | Strategy |
|-----------|----------|
| Small dataset (<10K examples) | **Freeze**. Not enough signal to safely move vectors. |
| Generic task (sentiment, news topics) | **Freeze** is often enough. |
| Domain-specific (medical, legal, finance, restaurant reviews) | **Fine-tune** -- domain meaning shifts. |
| Very large dataset (>1M examples) | **Fine-tune** -- enough data to safely re-learn. |

## Section 5: Evaluating Embedding Quality

In [ ]:
ANALOGIES = [
    ('paris', 'france', 'rome', 'italy'),
    ('paris', 'france', 'berlin', 'germany'),
    ('paris', 'france', 'madrid', 'spain'),
    ('paris', 'france', 'tokyo', 'japan'),
    ('paris', 'france', 'london', 'england'),
    ('king',  'queen',  'man',   'woman'),
    ('boy',   'girl',   'father','mother'),
    ('boy',   'girl',   'son',   'daughter'),
    ('go',    'went',   'do',    'did'),
    ('go',    'went',   'see',   'saw'),
]

def analogy_accuracy(kv, analogies, topn=5):
    correct, total = 0, 0
    for a, b, c, d in analogies:
        if any(w not in kv for w in (a, b, c, d)):
            continue
        try:
            preds = kv.most_similar(positive=[b, c], negative=[a], topn=topn)
        except KeyError:
            continue
        if d in [w for w, _ in preds]:
            correct += 1
        total += 1
    return correct, total

for label, kv in [('A: random', kv_random), ('B: frozen', kv_frozen), ('C: finetune', kv_finetune)]:
    c, t = analogy_accuracy(kv, ANALOGIES, topn=5)
    pct = 100 * c / max(1, t)
    print(f"{label:>12s} | analogy@top5 = {c}/{t}  ({pct:.0f}%)")

### Demo: t-SNE visualization

In [ ]:
def tsne_plot(kv, title, n_words=300, n_clusters=5, ax=None):
    if kv is None:
        return
    words = [w for w, _ in counter.most_common() if w in kv][:n_words]
    vectors = np.stack([kv[w] for w in words])
    km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10).fit(vectors)
    coords = TSNE(n_components=2, random_state=SEED, perplexity=30,
                  init='pca', learning_rate='auto').fit_transform(vectors)
    if ax is None:
        ax = plt.gca()
    ax.scatter(coords[:, 0], coords[:, 1], c=km.labels_, cmap='tab10', s=18, alpha=0.7)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
tsne_plot(kv_random,   'A: random init',     ax=axes[0])
tsne_plot(kv_frozen,   'B: frozen GloVe',    ax=axes[1])
tsne_plot(kv_finetune, 'C: fine-tuned GloVe', ax=axes[2])
plt.tight_layout(); plt.show()

### Lab 5.1 (SOLUTION): Compare neighbors of a domain word and a generic word

In [ ]:
# Solution: Lab 5.1
domain_word  = 'pizza'    # Yelp-heavy word
generic_word = 'science'  # Wikipedia-heavy word; rarely in restaurant reviews

print(f"\n===== DOMAIN WORD: '{domain_word}' =====")
show_neighbors('A: random  ', kv_random,   domain_word)
show_neighbors('B: frozen  ', kv_frozen,   domain_word)
show_neighbors('C: finetune', kv_finetune, domain_word)

print(f"\n===== GENERIC WORD: '{generic_word}' =====")
show_neighbors('A: random  ', kv_random,   generic_word)
show_neighbors('B: frozen  ', kv_frozen,   generic_word)
show_neighbors('C: finetune', kv_finetune, generic_word)

# Explanation:
# Expected results:
# - DOMAIN word ('pizza'): the fine-tuned model produces the most useful neighbors
#   ('pizzas', 'crust', 'slice', 'pepperoni') because Yelp reviews talk about pizza
#   constantly. Frozen GloVe gives encyclopedic neighbors ('italian', 'pasta'). Random
#   gives noise.
# - GENERIC word ('science'): frozen GloVe wins because GloVe was trained on Wikipedia
#   where 'science' is everywhere. Fine-tuned GloVe has *forgotten* a bit of GloVe's
#   general knowledge -- this is **catastrophic forgetting** in miniature. Random fails
#   completely since 'science' barely appears in Yelp.
# Trade-off: fine-tuning improves on-domain performance at the cost of off-domain
# generality. In production with a single downstream task this is usually a great
# trade; if you need general-purpose embeddings, freeze.

## Section 6: Wrap-up

Recipe summary:

1. Download a pretrained checkpoint (GloVe / Word2Vec / fastText / ...).
2. Build an aligned `(vocab_size, embedding_dim)` matrix.
3. Plug into `nn.Embedding.from_pretrained(matrix, freeze=True/False, padding_idx=0)`.
4. Decide: freeze (safe, small data) or fine-tune (powerful, more data, smaller LR).

### Self-check quiz (with answers)

**1. 200 documents: train from scratch, frozen GloVe, or fine-tuned GloVe?**

Frozen GloVe. From scratch fails because 200 documents will not teach the model what *king* means. Fine-tuning will severely overfit on 200 examples and erase GloVe's good initialization. Frozen gives you the GloVe knowledge for free and only the (small) downstream head needs to learn.

**2. Why does `padding_idx=0` require row 0 to be zeros?**

`padding_idx` only zeros the **gradient** for that row, not the **value**. If row 0 is non-zero, padding tokens contribute their (random) vector to whatever pooling you do downstream, adding noise to the pooled representation. Zeroing the row makes padding contribute nothing.

**3. After fine-tuning, `most_similar('cheap')` returns `['crappy', 'stale', 'mediocre']`. Bug?**

No, this is the expected behavior. In Yelp 1-star reviews *cheap* almost always describes low quality. Fine-tuning shifted the vector toward that cluster. If you wanted the GloVe meaning ("inexpensive, affordable"), you should have frozen it.

**4. GloVe 50d but `EMBEDDING_DIM = 100`. What error and where?**

When you hand the matrix to `nn.Embedding.from_pretrained`, PyTorch complains that the pretrained tensor's last dim (50) does not match the embedding's `embedding_dim` (100), which is a shape mismatch. (Or earlier, when you allocate `np.zeros((V, 100))` and try to assign a 50-d GloVe vector, numpy raises a broadcasting error.)

**5. Why filter by `requires_grad` for the frozen variant?**

Adam allocates internal state (1st and 2nd moment buffers) for every parameter you pass it. Frozen embeddings never receive gradients, so those buffers are wasted memory. Some PyTorch versions also emit a warning. `filter(lambda p: p.requires_grad, model.parameters())` cleanly excludes them.

### Optional / extra labs

1. **GloVe 50d**: switch to `https://www.dropbox.com/s/9zb5q1zg7tu8t5e/glove.6B.50d.txt?dl=1` and `EMBEDDING_DIM = 50`. Notice the analogy accuracy drops a bit but training is faster.
2. **4th variant**: random init with `LR_FINETUNE`. The lower LR slows random learning down; without a good init, you need more updates, not smaller ones.
3. **20-epoch fine-tune**: watch the *generic-word* `most_similar` quality degrade over epochs. This is catastrophic forgetting observed gradually.

## Congratulations!

You can now do **transfer learning for embeddings** in PyTorch, the foundation for every downstream NLP system. On to Notebook 7: from word vectors to **sentence vectors** with sentence-transformers.